In [10]:
import random

def canal_q_aire(mot_code, t):
    """
    Simule un canal q-aire ajoutant exactement t erreurs. 
    """
    n = len(mot_code)
    F = mot_code.base_ring()
    y = copy(mot_code)
    
    # Correction : Utilisation de random.sample pour choisir les indices
    positions_erreurs = random.sample(range(n), int(t))
    
    for i in positions_erreurs:
        # On s'assure que l'erreur est non nulle
        erreur = F.random_element()
        while erreur == 0:
            erreur = F.random_element()
        y[i] += erreur
        
    return y

In [11]:
def berlekamp_massey_decoder(u_received, x_support, k):
    n = len(x_support)
    t = (n - k) // 2
    F = u_received.base_ring()
    R.<X> = PolynomialRing(F)
    
    # 1. Construction du polynôme de support Pi(X) [cite: 32]
    Pi = R.prod([X - xi for xi in x_support])
    
    # 2. Construction du polynôme d'interpolation U(X) [cite: 26, 27]
    # Sage utilise l'interpolation de Lagrange en interne
    U = R.lagrange_polynomial(zip(x_support, u_received))
    
    # 3. Algorithme d'Euclide Étendu [cite: 34-45]
    A_prev, A_curr = R(0), R(1)
    B_prev, B_curr = Pi, U
    
    # Condition d'arrêt prématuré : deg(B) < n - t [cite: 53, 54]
    while B_curr.degree() >= n - t:
        Q, r = B_prev.quo_rem(B_curr)
        B_prev, B_curr = B_curr, r
        A_prev, A_curr = A_curr, A_prev - Q * A_curr
        
    f_decoded = B_curr // A_curr
    return f_decoded


In [13]:
def codeur_RS(message_coeffs, x_support):
    """
    Codeur Reed-Solomon : évalue le polynôme f (défini par message_coeffs)
    en chaque point du support x_support. 
    """
    F = x_support[0].parent()
    R.<X> = PolynomialRing(F)
    
    # On construit le polynôme f à partir des coefficients du message
    f = R(list(message_coeffs))
    
    # Le mot de code est le vecteur des évaluations (f(x1), ..., f(xn))
    c = vector(F, [f(xi) for xi in x_support])
    return c

In [15]:
# 1. Définition du corps
F.<a> = GF(256)

# 2. Définition de l'anneau des polynômes pour RS
R.<X> = PolynomialRing(F)

# 3. Paramètres du code RS (exemple n=255, k=223 pour le standard NASA)
n = 255
k = 223
t_cap = (n - k) // 2  # Capacité de correction : 16 erreurs
# 1. Création du support
x_pts = list(F)[:n]

# 2. Création d'un message aléatoire de longueur k
msg = [F.random_element() for _ in range(k)]

# 3. Encodage
mot_code = codeur_RS(msg, x_pts)

# 4. Passage dans le canal avec t = 3 erreurs (limite du code) 
mot_recu = canal_q_aire(mot_code, 3)

# 5. Décodage
f_retrouve = berlekamp_massey_decoder(mot_recu, x_pts, k)

# Vérification
print(f"Message original : {PolynomialRing(F, 'X')(msg)}")
print(f"Message décodé   : {f_retrouve}")
print(f"Succès du décodage : {f_retrouve == PolynomialRing(F, 'X')(msg)}")

Message original : (a^7 + a)*X^222 + (a^6 + a^5 + a^3 + a^2 + a + 1)*X^221 + (a^7 + a^6 + a^4 + a^3 + a^2 + a + 1)*X^220 + (a^7 + a^4 + a^3 + a^2 + a + 1)*X^219 + (a^7 + a^6 + a^5 + a^4 + a^2 + 1)*X^218 + (a^7 + a^6 + a^4 + a^3)*X^217 + (a^7 + a^6 + a^2 + a)*X^216 + (a^6 + a^5 + a^2 + 1)*X^215 + (a^7 + a^6 + a^3)*X^214 + (a^6 + a^4 + a^3 + a^2)*X^213 + (a^7 + a^6 + a^3)*X^212 + (a^7 + a^5 + a^4 + 1)*X^211 + (a^7 + a^4 + a^3 + a^2)*X^210 + a^5*X^209 + (a^3 + a^2 + 1)*X^208 + (a^7 + a^6 + a^4 + a^3 + a + 1)*X^207 + (a^6 + a^5 + a^2 + a)*X^206 + (a^7 + a^4 + a^2 + a)*X^205 + (a^6 + a^5 + a^4 + a^3)*X^204 + a^3*X^203 + (a^3 + a^2 + a)*X^202 + (a^7 + a^2 + a + 1)*X^201 + (a^3 + 1)*X^200 + (a^6 + a^5 + a)*X^199 + (a^6 + a^3 + a + 1)*X^198 + (a^7 + a^6 + a^4 + a^2)*X^197 + (a^2 + 1)*X^196 + (a^7 + a)*X^195 + (a^7 + a^5 + a)*X^194 + (a^6 + a^2)*X^193 + (a^5 + a^4 + a^3 + 1)*X^192 + (a^7 + a^2 + a)*X^191 + (a^5 + a^4 + a^3 + a^2)*X^190 + (a^4 + a + 1)*X^189 + (a^7 + a^5 + a^4 + a^3 + a + 1)*X^1